In [1]:
%load_ext autoreload
%autoreload 2

This notebook tackles the [**Playground Series – Season 5, Episode 12: Diabetes Prediction Challenge**](https://www.kaggle.com/competitions/playground-series-s5e12), a competition focused on aiding medical diagnosis: predict if a patient will be diagnosed with diabetes.  The goal is to develop a model that can accurately predict whether a patient will be diagnosed with diabetes, based on a set of medical and demographic features.

## Introduction

XGBoost is highly effective at capturing non-linear relationships between physiological markers (like BMI and Blood Pressure) and the target. It handles missing values internally (if any exist) and includes regularization (L1/L2) to prevent overfitting on smaller datasets.
We will need to encode our categorical variables (One-Hot or Label Encoding) before feeding them into XGBoost, as it treats inputs as numerical matrices.
The notebook is one of a number of notebooks that explore various models to address this challenge.  The EDA for this dataset was performed in this notebook: [**PS-S5E12: EDA**](https://www.kaggle.com/code/stephentarter/ps-s5e12-eda).

## Install Needed Packages

In [2]:
import os
import sys
import math
import random
import warnings
from pathlib import Path
from typing import Iterable
from IPython.display import display, Markdown, IFrame

# --- Third-party
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

from diabetes_preprocessing import FeatureFactory

# --- Notebook settings
warnings.filterwarnings('ignore')

%matplotlib inline

In [3]:
# Define some utilities functions
def configure_notebook(seed: int = 10301, float_precision: int = 3, max_columns: int = 15, max_rows: int = 25) -> int:
    """
    Configure notebook settings:
      - Disables warnings for cleaner output.
      - Sets pandas display options for better table formatting.
      - Returns a seed value for reproducibility.
    
    Parameters:
      seed (int): Random seed (default 548).
      float_precision (int): Number of decimal places for floats (default 3).
      max_columns (int): Maximum number of columns to display (default 15).
      max_rows (int): Maximum number of rows to display (default 25).

    Returns:
      int: The provided seed.
    """
    # Disable all warnings
    warnings.filterwarnings('ignore')
    
    # Set pandas display options for nicer output
    pd.options.display.float_format = f'{{:,.{float_precision}f}}'.format
    pd.set_option('display.max_columns', max_columns)
    pd.set_option('display.max_rows', max_rows)

    # Set seeds for reproducibility in numpy and the standard random module
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

def running_in_kaggle() -> bool:
    """
    Heuristics that are true in Kaggle notebooks:
    - Special directories exist (/kaggle/input, /kaggle/working)
    - Env var KAGGLE_KERNEL_RUN_TYPE is set
    - The kaggle_secrets module is available
    """
    try:
        if os.path.isdir('/kaggle/input') and os.path.isdir('/kaggle/working'):
            return True
        if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
            return True
        import kaggle_secrets  # noqa: F401  (only exists in Kaggle)
        return True
    except Exception:
        return False

In [4]:
# Apply configuration and set random seeds for reproducibility
seed = configure_notebook(max_columns = None, max_rows = None)

TARGET = 'diagnosed_diabetes'
USE_GPU = False

## Read and Examine the Training Dataset

In [5]:
DATA_DIR = Path('/kaggle/input/playground-series-s5e12') if running_in_kaggle() else Path('data')

training_df = pd.read_csv(DATA_DIR / 'train.csv')
print(training_df.head(5))

   id  age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
0   0   31                             1                                  45   
1   1   50                             2                                  73   
2   2   32                             3                                 158   
3   3   54                             3                                  77   
4   4   54                             1                                  55   

   diet_score  sleep_hours_per_day  screen_time_hours_per_day    bmi  \
0       7.700                6.800                      6.100 33.400   
1       5.700                6.500                      5.800 23.800   
2       8.500                7.400                      9.100 24.100   
3       4.600                7.000                      9.200 26.600   
4       5.700                6.200                      5.100 28.800   

   waist_to_hip_ratio  systolic_bp  diastolic_bp  heart_rate  \
0               0.930 

## Read and Examine the Test Dataset

In [6]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(test_df.head(5))

       id  age  alcohol_consumption_per_week  \
0  700000   45                             4   
1  700001   35                             1   
2  700002   45                             1   
3  700003   55                             2   
4  700004   77                             2   

   physical_activity_minutes_per_week  diet_score  sleep_hours_per_day  \
0                                 100       4.300                6.800   
1                                  87       3.500                4.600   
2                                  61       7.600                6.800   
3                                  81       7.300                7.300   
4                                  29       7.300                7.600   

   screen_time_hours_per_day    bmi  waist_to_hip_ratio  systolic_bp  \
0                      6.200 25.500               0.840          123   
1                      9.000 28.600               0.880          120   
2                      7.000 28.500               

## Model Training

In [7]:
# Define which strategies to use for the XGBoost model here:
fe_strategies = ['drop_id', 'ratios', 'log', 'polynomials', 'one_hot_encoding']

# Identify Feature Types
features = [c for c in training_df.columns if c != TARGET]

# Automatically select categorical columns for encoding
cat_features = training_df[features].select_dtypes(include=['object', 'category']).columns.tolist()

CONF = {
    'seed': seed,
    'n_folds': 5,
    'target': 'diagnosed_diabetes',
    'xgb_params': {
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'n_jobs': -1,
        'random_state': 42,
        'tree_method': 'gpu_hist' if USE_GPU else None,
    }
}

# Cross-Validation Training Loop
X = training_df[features]
y = training_df[CONF['target']]

# Arrays to store results
oof_preds = np.zeros(len(X))     # Out-of-Fold predictions (for blending later)
test_preds = np.zeros(len(test_df)) # Average test predictions

skf = StratifiedKFold(n_splits=CONF['n_folds'], shuffle=True, random_state=CONF['seed'])

print(f"\nStarting XGBoost Training ({CONF['n_folds']} Folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Transform Datakage)
    # Note: We are fitting the preprocessor every fold. 
    model_pipeline = Pipeline(steps=[
        ('feature_factory', FeatureFactory(strategies=fe_strategies, target=CONF['target'])),
        ('model', xgb.XGBClassifier(**CONF['xgb_params']))
    ]) 
    
    # Train Model
    # The pipeline will call feature_factory.fit_transform(X_train)
    # and then pass that result to xgb.fit()
    model_pipeline.fit(
        X_train, y_train,
    )
    
    # Predict Validation Set
    # [:, 1] grabs the probability of class 1 (Diabetes)
    val_probs = model_pipeline.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_probs
    
    # Score
    score = roc_auc_score(y_val, val_probs)
    print(f"Fold {fold+1} AUC: {score:.5f}")
    
    # Predict Test Set (Accumulate for averaging)
    test_probs = model_pipeline.predict_proba(test_df)[:, 1]
    test_preds += test_probs / CONF['n_folds']

# Evaluation

overall_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall OOF AUC: {overall_auc:.5f}")


Starting XGBoost Training (5 Folds)...
Fold 1 AUC: 0.72594
Fold 2 AUC: 0.72603
Fold 3 AUC: 0.72585
Fold 4 AUC: 0.72580
Fold 5 AUC: 0.72456

Overall OOF AUC: 0.72563


## Prepare Submission

In [8]:
submission_df = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission_df[TARGET] = test_preds
submission_df.head(10)

,id,diagnosed_diabetes
0,700000,0.471
1,700001,0.682
2,700002,0.773
3,700003,0.401
4,700004,0.930
5,700005,0.630
6,700006,0.737
7,700007,0.929
8,700008,0.572
9,700009,0.786


In [9]:
submission_df.to_csv("submission.csv", index=False)
print("Saved: submission.csv")

Saved: submission.csv
